# Лабораторная работа № 5
## Расчёт метрик и подготовка данных для анализа

**Проект:** Seoul Bike Sharing Demand  
**Группа:** ИВТ-241

### Аналитический вопрос

В какие часы, сезоны и типы дней наблюдается наиболее высокий и низкий спрос на аренду велосипедов?

### Цель работы

Рассчитать метрики спроса на аренду велосипедов на основе готовой аналитической витрины, проверить корректность полученных значений и подготовить результаты для дальнейшего анализа и визуализации.

### Источник данных

Аналитическая витрина `hourly_demand.parquet`, подготовленная в лабораторной работе № 4 и сохранённая в S3-хранилище SeaweedFS.

### Планируемые результаты

- Основная группировка метрик по часам.
- Уточняющая группировка по часам, сезонам и типам дней.
- Набор отдельных числовых наблюдений для изучения распределения количества аренд.
- Проверки корректности рассчитанных метрик.

In [1]:

from pathlib import Path
import os

import duckdb
from dotenv import load_dotenv

# Находим корень проекта и загружаем настройки.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in candidates if (path / ".env").is_file()),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Не найден .env в папке проекта.")

load_dotenv(PROJECT_ROOT / ".env")

# Путь к аналитической витрине из ЛР № 4.
INGESTED_ON = "2026-10-13"

MART_S3_PATH = (
    f"s3://mart/seoul_bike/"
    f"source_ingested_on={INGESTED_ON}/hourly_demand.parquet"
)

# Проверяем настройки доступа к S3.
S3_ENDPOINT = os.getenv("S3_ENDPOINT", "localhost:8333")
S3_ACCESS_KEY = os.getenv("S3_ACCESS_KEY")
S3_SECRET_KEY = os.getenv("S3_SECRET_KEY")

if not S3_ACCESS_KEY or not S3_SECRET_KEY:
    raise RuntimeError(
        "В .env должны быть заданы S3_ACCESS_KEY и S3_SECRET_KEY."
    )

# Подключаем DuckDB к SeaweedFS.
con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

s3_settings = {
    "s3_endpoint": S3_ENDPOINT,
    "s3_access_key_id": S3_ACCESS_KEY,
    "s3_secret_access_key": S3_SECRET_KEY,
    "s3_region": "us-east-1",
}

for name, value in s3_settings.items():
    escaped_value = value.replace("'", "''")
    con.execute(f"SET {name} = '{escaped_value}';")

con.execute("SET s3_use_ssl = false;")
con.execute("SET s3_url_style = 'path';")

# Источник для последующих SQL-запросов.
escaped_mart_path = MART_S3_PATH.replace("'", "''")
mart_file = (
    f"read_parquet('{escaped_mart_path}', hive_partitioning=true)"
)

# Проверяем чтение сохранённой витрины.
mart_rows = con.execute(
    f"SELECT COUNT(*) FROM {mart_file}"
).fetchone()[0]

mart_schema = con.execute(
    f"DESCRIBE SELECT * FROM {mart_file}"
).fetchall()

print("Корень проекта:", PROJECT_ROOT)
print("Витрина:", MART_S3_PATH)
print("Количество строк:", mart_rows)

print("\nСхема витрины:")
for column in mart_schema:
    print(f"  {column[0]}: {column[1]}")


Корень проекта: c:\Users\Даниил\Desktop\Учусь типа\Семестр 5\ИТАД\SeoulBikeLab2
Витрина: s3://mart/seoul_bike/source_ingested_on=2026-10-13/hourly_demand.parquet
Количество строк: 8760

Схема витрины:
  observation_date: DATE
  hour: BIGINT
  rented_bike_count: BIGINT
  season: VARCHAR
  holiday: VARCHAR
  functioning_day: BOOLEAN
  iso_weekday: INTEGER
  day_of_week: VARCHAR
  day_type: VARCHAR
  temperature_c: DOUBLE
  humidity_pct: BIGINT
  wind_speed_m_s: DOUBLE
  visibility_10m: BIGINT
  dew_point_temperature_c: DOUBLE
  solar_radiation_mj_m2: DOUBLE
  rainfall_mm: DOUBLE
  snowfall_cm: DOUBLE
  quality_flag_humidity_out_of_range: BOOLEAN
  source_ingested_on: DATE


In [2]:

# 1. Общие метрики по всей витрине.

overall_metrics = con.execute(
    f"""
    SELECT
        COUNT(*) AS observation_count,
        SUM(rented_bike_count) AS total_rentals,
        COUNT(*) FILTER (
            WHERE functioning_day = TRUE
        ) AS operating_observation_count,
        SUM(rented_bike_count) FILTER (
            WHERE functioning_day = TRUE
        ) AS operating_rentals_total,
        AVG(rented_bike_count) FILTER (
            WHERE functioning_day = TRUE
        ) AS avg_rentals_operating
    FROM {mart_file}
    """
).fetchone()

print("Общие метрики")
print("Всего почасовых наблюдений:", overall_metrics[0])
print("Всего аренд:", overall_metrics[1])
print("Наблюдений при работающей системе:", overall_metrics[2])
print("Аренд за часы работы системы:", overall_metrics[3])
print("Среднее количество аренд за час работы системы:",
      round(overall_metrics[4], 2))


# 2. Основная группировка по часам.
# Временная таблица остаётся доступной в текущем соединении DuckDB.

con.execute(
    f"""
    CREATE OR REPLACE TEMP TABLE metrics_by_hour AS
    WITH hourly AS (
        SELECT
            hour,
            COUNT(*) AS observation_count,
            SUM(rented_bike_count) AS total_rentals,

            COUNT(*) FILTER (
                WHERE functioning_day = TRUE
            ) AS operating_observation_count,

            SUM(rented_bike_count) FILTER (
                WHERE functioning_day = TRUE
            ) AS operating_rentals_total,

            AVG(rented_bike_count) FILTER (
                WHERE functioning_day = TRUE
            ) AS avg_rentals_operating

        FROM {mart_file}
        GROUP BY hour
    )
    SELECT
        hour,
        observation_count,
        total_rentals,
        operating_observation_count,
        operating_rentals_total,
        avg_rentals_operating,

        -- Доля часа в общем количестве аренд.
        total_rentals / NULLIF(
            SUM(total_rentals) OVER (),
            0
        ) AS share_of_total_rentals

    FROM hourly
    """
)

# Показываем результаты основной группировки.
hourly_results = con.execute(
    """
    SELECT
        hour,
        observation_count,
        total_rentals,
        operating_observation_count,
        ROUND(avg_rentals_operating, 2) AS avg_rentals_operating,
        ROUND(100 * share_of_total_rentals, 2) AS share_pct
    FROM metrics_by_hour
    ORDER BY hour
    """
).fetchall()

print("\nМетрики по часам")
print(
    f"{'Час':>4} {'Наблюдений':>12} {'Всего аренд':>13} "
    f"{'Часов работы':>14} {'Среднее аренд':>15} {'Доля, %':>10}"
)

for row in hourly_results:
    print(
        f"{row[0]:>4} {row[1]:>12} {row[2]:>13} "
        f"{row[3]:>14} {row[4]:>15.2f} {row[5]:>10.2f}"
    )

print("\nОсновная группировка сохранена в DuckDB: metrics_by_hour")


Общие метрики
Всего почасовых наблюдений: 8760
Всего аренд: 6172314
Наблюдений при работающей системе: 8465
Аренд за часы работы системы: 6172314
Среднее количество аренд за час работы системы: 729.16

Метрики по часам
 Час   Наблюдений   Всего аренд   Часов работы   Среднее аренд    Доля, %
   0          365        197633            352          561.46       3.20
   1          365        155557            352          441.92       2.52
   2          365        110095            352          312.77       1.78
   3          365         74216            352          210.84       1.20
   4          365         48396            352          137.49       0.78
   5          365         50765            352          144.22       0.82
   6          365        104961            352          298.18       1.70
   7          365        221192            353          626.61       3.58
   8          365        370731            353         1050.23       6.01
   9          365        235784          

In [3]:

# Уточняющая группировка:
# час + сезон + тип дня.

con.execute(
    f"""
    CREATE OR REPLACE TEMP TABLE metrics_by_hour_season_daytype AS
    SELECT
        hour,
        season,
        day_type,

        COUNT(*) AS observation_count,

        SUM(rented_bike_count) AS total_rentals,

        COUNT(*) FILTER (
            WHERE functioning_day = TRUE
        ) AS operating_observation_count,

        SUM(rented_bike_count) FILTER (
            WHERE functioning_day = TRUE
        ) AS operating_rentals_total,

        AVG(rented_bike_count) FILTER (
            WHERE functioning_day = TRUE
        ) AS avg_rentals_operating

    FROM {mart_file}
    GROUP BY
        hour,
        season,
        day_type
    """
)

# Выводим срезы с самым высоким и самым низким средним спросом.
refined_results = con.execute(
    """
    SELECT
        hour,
        season,
        day_type,
        observation_count,
        operating_observation_count,
        total_rentals,
        ROUND(avg_rentals_operating, 2) AS avg_rentals_operating
    FROM metrics_by_hour_season_daytype
    WHERE operating_observation_count > 0
    ORDER BY avg_rentals_operating DESC
    """
).fetchall()

print("Уточняющая группировка: топ-10 срезов по среднему спросу")
print(
    f"{'Час':>4} {'Сезон':<10} {'Тип дня':<10} "
    f"{'Наблюдений':>12} {'Работа системы':>15} "
    f"{'Всего аренд':>13} {'Среднее':>10}"
)

for row in refined_results[:10]:
    print(
        f"{row[0]:>4} {row[1]:<10} {row[2]:<10} "
        f"{row[3]:>12} {row[4]:>15} "
        f"{row[5]:>13} {row[6]:>10.2f}"
    )

print("\nУточняющая группировка сохранена в DuckDB:")
print("metrics_by_hour_season_daytype")
print("Число групп:", len(refined_results))


Уточняющая группировка: топ-10 срезов по среднему спросу
 Час Сезон      Тип дня      Наблюдений  Работа системы   Всего аренд    Среднее
  18 Summer     Weekday              66              66        153606    2327.36
  18 Autumn     Weekday              65              57        125454    2200.95
  19 Summer     Weekday              66              66        128734    1950.52
  18 Spring     Weekday              66              64        120820    1887.81
  20 Summer     Weekday              66              66        119990    1818.03
  21 Summer     Weekday              66              66        117865    1785.83
  20 Summer     Weekend              26              26         45787    1761.04
  19 Summer     Weekend              26              26         45077    1733.73
   8 Summer     Weekday              66              66        113573    1720.80
   8 Autumn     Weekday              65              57         96222    1688.11

Уточняющая группировка сохранена в DuckDB:
metrics_

In [4]:

# Подготавливаем отдельные числовые наблюдения
# для будущего анализа распределения количества аренд.

con.execute(
    f"""
    CREATE OR REPLACE TEMP TABLE metrics_rentals_distribution AS
    SELECT
        observation_date,
        hour,
        rented_bike_count
    FROM {mart_file}
    WHERE functioning_day = TRUE
      AND rented_bike_count IS NOT NULL
    """
)

# Проверяем подготовленную выборку.
distribution_summary = con.execute(
    """
    SELECT
        COUNT(*) AS observation_count,
        COUNT(rented_bike_count) AS valid_values,
        MIN(rented_bike_count) AS min_rentals,
        MAX(rented_bike_count) AS max_rentals,
        ROUND(AVG(rented_bike_count), 2) AS avg_rentals,
        MEDIAN(rented_bike_count) AS median_rentals
    FROM metrics_rentals_distribution
    """
).fetchone()

print("Распределение количества аренд")
print("Всего наблюдений:", distribution_summary[0])
print("Заполненных числовых значений:", distribution_summary[1])
print("Минимум аренд:", distribution_summary[2])
print("Максимум аренд:", distribution_summary[3])
print("Среднее количество аренд:", distribution_summary[4])
print("Медиана количества аренд:", distribution_summary[5])

print("\nВременная таблица создана:")
print("metrics_rentals_distribution")


Распределение количества аренд
Всего наблюдений: 8465
Заполненных числовых значений: 8465
Минимум аренд: 2
Максимум аренд: 3556
Среднее количество аренд: 729.16
Медиана количества аренд: 542.0

Временная таблица создана:
metrics_rentals_distribution


In [5]:

import math

# Проверка 1. Сверяем основную группировку с общими итогами.

hourly_totals = con.execute("""
SELECT
    SUM(observation_count),
    SUM(total_rentals),
    SUM(operating_observation_count),
    SUM(operating_rentals_total)
FROM metrics_by_hour
""").fetchone()

print("Проверка 1. Сверка основной группировки")
print("Все наблюдения:", hourly_totals[0], "| Ожидалось:", overall_metrics[0])
print("Все аренды:", hourly_totals[1], "| Ожидалось:", overall_metrics[1])
print("Часы работы:", hourly_totals[2], "| Ожидалось:", overall_metrics[2])
print("Аренды за часы работы:", hourly_totals[3], "| Ожидалось:", overall_metrics[3])

assert hourly_totals[0] == overall_metrics[0]
assert hourly_totals[1] == overall_metrics[1]
assert hourly_totals[2] == overall_metrics[2]
assert hourly_totals[3] == overall_metrics[3]

print("Результат: проверка пройдена.")


# Проверка 2. Сумма долей всех часов должна составлять 100%.

share_total = con.execute("""
SELECT SUM(share_of_total_rentals)
FROM metrics_by_hour
""").fetchone()[0]

share_percent = share_total * 100

print("\nПроверка 2. Сумма долей по часам")
print("Полученная сумма долей:", round(share_percent, 6), "%")

assert math.isclose(
    share_total,
    1.0,
    rel_tol=1e-9,
    abs_tol=1e-9
)

print("Результат: проверка пройдена.")


# Проверка 3. Сверяем количество отдельных наблюдений
# с количеством работающих часов в исходной витрине.

distribution_count = con.execute("""
SELECT COUNT(*)
FROM metrics_rentals_distribution
""").fetchone()[0]

source_operating_count = con.execute(
    f"""
    SELECT COUNT(*)
    FROM {mart_file}
    WHERE functioning_day = TRUE
      AND rented_bike_count IS NOT NULL
    """
).fetchone()[0]

print("\nПроверка 3. Число отдельных числовых наблюдений")
print("В подготовленной выборке:", distribution_count)
print("В исходной витрине:", source_operating_count)

assert distribution_count == source_operating_count
assert distribution_count == overall_metrics[2]

print("Результат: проверка пройдена.")

print("\nВсе три проверки выполнены успешно.")


Проверка 1. Сверка основной группировки
Все наблюдения: 8760 | Ожидалось: 8760
Все аренды: 6172314 | Ожидалось: 6172314
Часы работы: 8465 | Ожидалось: 8465
Аренды за часы работы: 6172314 | Ожидалось: 6172314
Результат: проверка пройдена.

Проверка 2. Сумма долей по часам
Полученная сумма долей: 100.0 %
Результат: проверка пройдена.

Проверка 3. Число отдельных числовых наблюдений
В подготовленной выборке: 8465
В исходной витрине: 8465
Результат: проверка пройдена.

Все три проверки выполнены успешно.
